This notebook builds stacked histograms using VAMP-seq and SGE data seen in figure 2.

In [ ]:
import pandas as pd
import altair as alt
import numpy as np

In [ ]:
#Paths to each of the SGE and VAMP-seq subsets from the main dataframe
sge_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251202_SGEsubset.xlsx')
vampseq_ppj_subset = pd.read_excel('../Data/filtered_ppj_data/20251204_VAMPseqsubset_wDups.xlsx')

alt.data_transformers.disable_max_rows() #gets rid of max data length problem

In [ ]:
def get_gmm_threshold(df): #estimates thresholds by finding highest scoring/lowest scoring functionally abnormal/functionally normal variants respectively
    
    ab_df = df.loc[df['functional_consequence'].isin(['functionally_abnormal'])]
    norm_df = df.loc[df['functional_consequence'].isin(['functionally_normal'])]

    # now we get the scores that are the closest to the (n)ormal and (a)bnormal thresholds
    upprthresh = norm_df['score'].min()
    lwrthresh = ab_df['score'].max()


    thresholds = [lwrthresh, upprthresh]

    return thresholds

In [ ]:
def read_data(sge, vampseq): #Reads all data and changes variant consequence annotations to be more human readable
    
    sge_genes = ['BARD1', 'PALB2', 'BRCA2', 'RAD51D', 'XRCC2', 'CTCF', 'SFPQ'] #SGE genes

    vampseq_genes = ['G6PD', 'TSC2', 'F9'] #VAMP-seq genes

    sge_dict = {} #Empty dictionary to store SGE data in form or {gene_name: gene_data}

    for gene in sge_genes:
        print(f' Reading {gene}...')
        df = sge.loc[sge['Gene'] == gene]

        df = df.dropna(subset = ['simplified_consequence'])
        df = df.rename(columns = {'simplified_consequence': 'Consequence',
                                 'auth_reported_score': 'score',
                                  'auth_reported_func_class': 'functional_consequence',
                                 'ref_allele': 'ref',
                                 'clinvar_sig': 'Germline classification'})
        df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
        df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
        df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
        df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
        df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
        df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
        df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
        df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
        df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'
        df.loc[df['ref'].str.len() > 1, 'Consequence'] = '3bp Deletion'

        sge_dict[gene] = df


    #Analagous code for reading VAMP-seq data
    vamp_dict = {}
    for gene in vampseq_genes:
        #Merges TSC2 datasets into single dataframe
        if gene == 'TSC2': 
            df = vampseq.loc[vampseq['Gene'] == gene]

            rapgap_df = df.loc[df['Dataset'].str.contains('rapgap')].copy()
            tuberin_df = df.loc[df['Dataset'].str.contains('tuberin')].copy()
            tsc2_sets = [('TSC2 RAP-GAP', rapgap_df), ('TSC2 Tuberin', tuberin_df)]

            for set in tsc2_sets:
                name, df = set
                df['simplified_consequence'] = df['simplified_consequence'].fillna('filter')
                df = df.rename(columns = {'simplified_consequence': 'Consequence',
                                         'auth_reported_score': 'score',
                                          'auth_reported_func_class': 'functional_consequence',
                                         'ref_allele': 'ref',
                                         'clinvar_sig': 'Germline classification'})
                df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
                df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
                df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
                df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
                df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
                df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
                df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
                df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
                df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'

                vamp_dict[name] = df
        else:
            df = vampseq.loc[vampseq['Gene'] == gene]
            df['simplified_consequence'] = df['simplified_consequence'].fillna('filter')
            df = df.rename(columns = {'simplified_consequence': 'Consequence',
                                     'auth_reported_score': 'score',
                                      'auth_reported_func_class': 'functional_consequence',
                                     'ref_allele': 'ref',
                                     'clinvar_sig': 'Germline classification'})
            df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
            df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
            df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
            df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
            df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
            df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
            df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
            df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
            df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'
        
        vamp_dict[gene] = df

    return sge_dict, vamp_dict

In [ ]:
def clinvar_density_plot(df,plot_domain): #Custom function to build ClinVar density plot overlay onto histograms
    
    w_clinvar_df = (df.dropna(subset = ['Germline classification'])).copy() #Gets ClinVar variants
    
    w_clinvar_df = w_clinvar_df.loc[~w_clinvar_df['Germline classification'].isin(['Uncertain significance', 'Conflicting classifications of pathogenicity'])] #Drops VUS

    #Renames BLB and PLP variant types. All BLB and PLPs are grouped into benign and pathogenic groups respectively
    w_clinvar_df.loc[(w_clinvar_df['Germline classification'] == 'Benign') | (w_clinvar_df['Germline classification'] == 'Likely benign') | (w_clinvar_df['Germline classification'] == 'Benign/Likely benign'), 'simple_classification'] = 'Benign/Likely Benign'
    w_clinvar_df.loc[(w_clinvar_df['Germline classification'] == 'Pathogenic') | (w_clinvar_df['Germline classification'] == 'Likely pathogenic') | (w_clinvar_df['Germline classification'] == 'Pathogenic/Likely pathogenic'), 'simple_classification'] = 'Pathogenic/Likely Pathogenic'
    
    w_clinvar_df = w_clinvar_df.loc[w_clinvar_df['simple_classification'].isin(['Benign/Likely Benign',  'Pathogenic/Likely Pathogenic'])]

    #Builds density plot
    print(f' This is how many ClinVar variants there are: {len(w_clinvar_df)}')
    density_plot = alt.Chart(w_clinvar_df).transform_density(
        'score',
        as_=['score', 'density'],
        groupby=['simple_classification'],
        extent=plot_domain
    ).mark_line(strokeWidth=2).encode(
        x=alt.X('score:Q', scale=alt.Scale(domain=plot_domain)),
        y=alt.Y('density:Q', axis = None),
        color=alt.Color('simple_classification:N',
                       scale = alt.Scale(
                           domain = ['Pathogenic/Likely Pathogenic', 'Benign/Likely Benign'],
                           range = ['#CA7682','#1D7AAB']
                       ),
                      legend = None
                       )
    )

    #Extra plot for legend 
    w_legend_density_plot = alt.Chart(w_clinvar_df).transform_density(
        'score',
        as_=['score', 'density'],
        groupby=['simple_classification'],
        extent=plot_domain
    ).mark_line(strokeWidth=2).encode(
        x=alt.X('score:Q', scale=alt.Scale(domain=plot_domain)),
        y=alt.Y('density:Q', axis = None),
        color=alt.Color('simple_classification:N',
                       scale = alt.Scale(
                           domain = ['Pathogenic/Likely Pathogenic', 'Benign/Likely Benign'],
                           range = ['#CA7682','#1D7AAB']
                       ),
                        legend = alt.Legend(
                            title = 'ClinVar Classification',
                            titleFontSize = 14,
                            labelFontSize = 12
                        )
        )
    )

    #w_legend_density_plot.display()
    return density_plot

In [ ]:
def sge_stacked_histos(sge_genes, variable_bins=False): #Builds stacked histograms for SGE genes

    genes = list(sge_genes.keys())

    #Dimensions of each individual plot
    std_height = 150
    std_width = 500
    std_dy = 15
    
    palette = [
        '#006616',  # dark green
        '#81B4C7',  # dusty blue
        '#ffcd3a',  # yellow
        '#6AA84F',  # med green
        '#93C47D',  # light green
        '#888888',  # med gray
        '#000000',  # black
        '#1170AA',  # darker blue
        '#CFCFCF'   # light gray  
    ]
    
    variant_types = [
        'Synonymous',
        'Missense',  
        'Stop Gained',
        'Intron', 
        'UTR Variant',
        'Stop Lost',
        'Start Lost',
        'Canonical Splice', 
        'Splice Region'
    ]

    histograms = []

    plot_domain = [-0.45, 0.1]
    tick_values = [-0.4, -0.3, -0.2, -0.1, 0, 0.1]
    title_fontsize = 26
    
    for i, gene in enumerate(genes):
        df = sge_genes[gene]
        length = len(df)

        # Filter to plot domain
        df = df.loc[(df['score'] >= plot_domain[0]) & (df['score'] <= plot_domain[1])].copy()
        
        # Determine thresholds
        if gene != 'BRCA2':
            thresholds = get_gmm_threshold(df)
            cutoff = thresholds[0]
        else:
            thresholds = None
            cutoff = -0.1
        
        # --- Build axis configurations based on position in list ---
        
        if i == 0:
            # First gene: include legend, no x-axis labels
            x_axis = alt.Axis(title='', labels=False, tickSize=3, tickWidth=3, values=tick_values)
            y_axis = alt.Axis(title='# of Vars.', labelFontSize=22, tickSize=3, tickWidth=3, titleFontSize=24)
            show_legend = alt.Legend(titleFontSize=18, labelFontSize=16, orient='top-left', columns=2, offset=10)
            
        elif i == len(genes) - 1:
            # Last gene: x-axis labels, no legend
            x_axis = alt.Axis(title='Fitness Score', titleFontSize=24, labelFontSize=22, tickSize=3, tickWidth=3, values=tick_values)
            y_axis = alt.Axis(title='', labelFontSize=22, titleFontSize=24)
            show_legend = None
            
        else:
            # Middle genes: no x-axis labels, no legend
            x_axis = alt.Axis(title='', labels=False, ticks=True, tickSize=3, tickWidth=3, values=tick_values)
            y_axis = alt.Axis(title='', tickCount=4, tickMinStep=1, labelFontSize=22, titleFontSize=24)
            show_legend = None
        
        # --- Build histogram based on variable_bins setting. If true, allows for larger bins for LoF variants only ---
        
        if variable_bins:
            # Layered approach: two histograms with different bin widths
            
            # Clamp edges to plot domain
            left_edge = max(df['score'].min(), plot_domain[0])
            right_edge = min(df['score'].max(), plot_domain[1])
            
            # Calculate step sizes: 5 bins below cutoff, 10 bins above
            low_step = (cutoff - left_edge) / 5
            high_step = (right_edge - cutoff) / 10
            
            # Split data at cutoff
            df_low = df[df['score'] < cutoff]
            df_high = df[df['score'] >= cutoff]
            
            # Low score histogram (5 bins)
            low_chart = alt.Chart(df_low).mark_bar().encode(
                alt.X('score:Q', 
                      axis=x_axis,
                      scale=alt.Scale(domain=plot_domain),
                      bin=alt.Bin(extent=[left_edge, cutoff], step=low_step)),
                alt.Y('count():Q', 
                      axis=y_axis,
                      scale=alt.Scale(zero=False)),
                color=alt.Color('Consequence:N', 
                                scale=alt.Scale(range=palette, domain=variant_types), 
                                legend=show_legend)
            )
            
            # High score histogram (10 bins)
            high_chart = alt.Chart(df_high).mark_bar().encode(
                alt.X('score:Q', 
                      axis=x_axis,
                      scale=alt.Scale(domain=plot_domain),
                      bin=alt.Bin(extent=[cutoff, right_edge], step=high_step)),
                alt.Y('count():Q', 
                      axis=y_axis,
                      scale=alt.Scale(zero=False)),
                color=alt.Color('Consequence:N', 
                                scale=alt.Scale(range=palette, domain=variant_types), 
                                legend=show_legend)
            )
            
            # Combine low and high histograms
            chart = (low_chart + high_chart).properties(
                height=std_height,
                width=std_width,
                title=alt.TitleParams(text=f' {gene} (n = {length})',
                                      fontSize=title_fontsize,
                                      dy=std_dy)
            )
        
        else:
            # Uniform binning: single histogram with consistent bin width
            chart = alt.Chart(df).mark_bar().encode(
                alt.X('score:Q', 
                      axis=x_axis,
                      scale=alt.Scale(domain=plot_domain),
                      bin=alt.Bin(maxbins=50)),
                alt.Y('count():Q', 
                      axis=y_axis,
                      scale=alt.Scale(zero=False)),
                color=alt.Color('Consequence:N', 
                                scale=alt.Scale(range=palette, domain=variant_types), 
                                legend=show_legend)
            ).properties(
                height=std_height,
                width=std_width,
                title=alt.TitleParams(text=f' {gene} (n = {length:,d})',
                                      fontSize=title_fontsize,
                                      dy=std_dy)
            )
        
        # --- Add threshold lines and density plots ---
        
        layers = [chart]
        
        # Add threshold lines (except for BRCA2)
        if thresholds is not None:
            nf_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[0]]})).mark_rule(
                color='black', strokeDash=[8, 8], strokeWidth=2
            ).encode(x='Functional Score:Q')
            
            func_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[1]]})).mark_rule(
                color='#888888', strokeDash=[8, 8], strokeWidth=2
            ).encode(x='Functional Score:Q')
            
            layers.extend([nf_line, func_line])
        
        # Add ClinVar density plot (except for last gene)
        if i != len(genes) - 1:
            clinvar_density = clinvar_density_plot(df, plot_domain)
            layers.append(clinvar_density)
            
            chart = alt.layer(*layers).resolve_scale(
                y='independent',
                color='independent'
            )
        else:
            chart = alt.layer(*layers)
        
        histograms.append(chart)

    # Combine all histograms vertically
    final_plot = histograms[0]
    for j in range(1, len(histograms)):
        final_plot = alt.vconcat(final_plot, histograms[j], spacing=-5)

    final_plot = final_plot.configure_axis(
        grid=False
    ).configure_view(
        stroke=None
    ).resolve_scale(
        x='shared',
        y='independent'
    )

    return final_plot

In [ ]:
def vampseq_stacked_histos(vampseq_data, thresholding = False): #Stacked histograms for VAMP-seq genes
    
    genes = list(vampseq_data.keys())

    histograms = []
    

    std_height = 262.5
    std_width = 500
    i = 0
    palette = [
        '#006616', # dark green,
        '#81B4C7', # dusty blue
        '#ffcd3a' # yellow  
    ]
    
    
    variant_types = [
        'Synonymous',
        'Missense',  
        'Stop Gained'
    ]
    plot_domain = [-0.25, 1.5]
    tick_values = [-0.25, 0, 0.25, 0.50, 0.75, 1, 1.25, 1.5]
    title_fontsize = 26

    i = 0
    
    while i < len(genes):
        gene = genes[i]
        df_dup = vampseq_data[gene]

        df = df_dup.drop_duplicates(subset = ['ID'])
        
        length = len(df)

        df = df.dropna(subset = ['Consequence'])
        df = df.loc[(df['score'] >= plot_domain[0]) & (df['score'] <= plot_domain[1])]
        thresholds = get_gmm_threshold(df)

        
        if gene == 'F9 Ab001' or gene == 'F9 Ab124' or gene == 'F9 Ab3570' or gene == 'F9 Strep':
            i += 1
            continue
        elif gene == 'G6PD':
            chart = chart = alt.Chart(df).mark_bar().encode(
                alt.X('score', 
                      axis = alt.Axis(title = '', 
                                      labels = False,
                                      tickSize = 3,
                                      tickWidth = 3,
                                      values= tick_values
                                     ), 
                      scale = alt.Scale(domain = plot_domain),
                      bin = alt.Bin(maxbins = 50)),
                alt.Y('count()', 
                      axis = alt.Axis(title = '# of Vars.', 
                                      labelFontSize = 22, 
                                      titleFontSize = 24)),
                color = alt.Color('Consequence:N', 
                             scale = alt.Scale(range = palette,
                                              domain = variant_types), 
                             legend = alt.Legend(titleFontSize = 18, 
                                                 labelFontSize = 16,
                                                 orient = 'top-left',
                                                 columns = 1,
                                                offset = 10))
            ).properties(
                height = std_height,
                width = std_width,
                title = alt.TitleParams(text = f' {gene} (n = {length:,d})',
                                        fontSize = title_fontsize,
                                        dy = 15
                                       )
            )

            clinvar_density = clinvar_density_plot(df, plot_domain)

            chart = alt.layer(chart,clinvar_density).resolve_scale(
                y = 'independent',
                color = 'independent'
                )

            if thresholding:
                nf_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[0]]})).mark_rule(color = 'black', strokeDash = [8,8], strokeWidth = 2).encode(
                x = 'Functional Score')
        
                func_line = alt.Chart(pd.DataFrame({'Functional Score': [thresholds[1]]})).mark_rule(color = '#888888', strokeDash = [8,8], strokeWidth = 2).encode(
                x = 'Functional Score')
        
        
                chart = alt.layer(chart, nf_line, func_line)
                
            histograms.append(chart)
        elif gene == 'F9':
            chart = alt.Chart(df).mark_bar().encode(
                alt.X('score', 
                      axis = alt.Axis(title = 'Abundance Score', 
                                  titleFontSize = 24,
                                 labelFontSize = 22,
                                 tickSize = 3,
                                tickWidth = 3,
                                 values = tick_values),
                      scale = alt.Scale(domain = plot_domain),
                      bin = alt.Bin(maxbins = 50)),
                alt.Y('count()', 
                      axis = alt.Axis(title = '', 
                                      labelFontSize = 22, 
                                      titleFontSize = 24)),
                color = alt.Color('Consequence:N', 
                             scale = alt.Scale(range = palette,
                                              domain = variant_types), 
                             legend = None)
            ).properties(
                height =std_height,
                width = std_width,
                title = alt.TitleParams(text = f' F9 (Heavy-Chain Ab) (n = {length:,d})',
                                        fontSize = title_fontsize,
                                        dy = 15
                                       )
            )
            clinvar_density = clinvar_density_plot(df, plot_domain)

            chart = alt.layer(chart,clinvar_density).resolve_scale(
                y = 'independent',
                color = 'independent'
                )

            if thresholding: 
                nf_line = alt.Chart(pd.DataFrame({'Abundance Score': [thresholds[0]]})).mark_rule(color = 'black', strokeDash = [8,8], strokeWidth = 2).encode(
                x = 'Abundance Score')
        
                func_line = alt.Chart(pd.DataFrame({'Abundance Score': [thresholds[1]]})).mark_rule(color = '#888888', strokeDash = [8,8], strokeWidth = 2).encode(
                x = 'Abundance Score')
        
        
                chart = alt.layer(chart, nf_line, func_line)
            histograms.append(chart)
        elif gene == 'TSC2 RAP-GAP' or gene == 'TSC2 Tuberin': 
            chart = alt.Chart(df).mark_bar().encode(
                alt.X('score', 
                      axis = alt.Axis(title = '',
                                      labels = False,
                                      ticks = True,
                                      tickSize = 3,
                                      tickWidth = 3,
                                      values = tick_values
                                     ),
                      scale = alt.Scale(domain = plot_domain),
                      bin = alt.Bin(maxbins = 50)),
                alt.Y('count()', 
                      axis = alt.Axis(title = '',
                                      tickCount = 3, 
                                      tickMinStep = 1,
                                      labelFontSize = 22, 
                                      titleFontSize = 24),
                     scale = alt.Scale(nice = False, zero = False)
                     ),
                color = alt.Color('Consequence:N', 
                             scale = alt.Scale(range = palette,
                                              domain = variant_types), 
                             legend = None)
            ).properties(
            height = std_height,
            width = std_width,
            title = alt.TitleParams(text = f' {gene} (n = {length:,d})',
                                    fontSize = title_fontsize,
                                    dy = 15
                                   )
            )
            clinvar_density = clinvar_density_plot(df, plot_domain)

            chart = alt.layer(chart,clinvar_density).resolve_scale(
                y = 'independent',
                color = 'independent'
                )

            if thresholding:
                nf_line = alt.Chart(pd.DataFrame({'Abundance Score': [thresholds[0]]})).mark_rule(color = 'black', strokeDash = [8,8], strokeWidth = 2).encode(
                x = 'Abundance Score')
        
                func_line = alt.Chart(pd.DataFrame({'Abundance Score': [thresholds[1]]})).mark_rule(color = '#888888', strokeDash = [8,8], strokeWidth = 2).encode(
                x = 'Abundance Score')
        
        
                chart = alt.layer(chart, nf_line, func_line)
        
            histograms.append(chart)
        i += 1


    j = 1
    while j < len(histograms):
        if j == 1:
            final_plot = alt.vconcat(histograms[0], histograms[j], spacing = -5)
        else: 
            final_plot = final_plot & histograms[j]
        j += 1


    final_plot = final_plot.configure_axis(
        grid = False
    ).configure_view(
        stroke = None
    ).resolve_scale(
        x = 'shared',
        y = 'independent'
    )

    final_plot.display()
    return final_plot

In [ ]:
def main():
    sge_data, vampseq_data = read_data(sge_ppj_subset, vampseq_ppj_subset)

    #Filters out 3bp-deletions in the XRCC2 dataset
    new_xrcc2 = sge_data['XRCC2']
    new_xrcc2 = new_xrcc2.loc[new_xrcc2['ref'].str.len() == 1]
    sge_data['XRCC2'] = new_xrcc2

    stacked_histos = sge_stacked_histos(sge_data)
    stacked_histos.display()
    
    vampseq_histos = vampseq_stacked_histos(vampseq_data, thresholding = True)
    
    save = False
    if save:
        stacked_histos.save('/Users/ivan/Desktop/pillar_project_figs/Histogram_wStripplot/20260114all_sge_histograms.svg')
        vampseq_histos.save('/Users/ivan/Desktop/pillar_project_figs/Histogram_wStripplot/20260114all_vampseq_histograms_bigger.svg')


In [ ]:
main()